<a href="https://colab.research.google.com/github/rmadatt/ADLAB/blob/main/Agriculture_Stablecoins_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==============================================================================
# INTERACTIVE FX SLIPPAGE & SETTLEMENT SAVINGS CALCULATOR FOR AGRI-EXPORTERS
# ==============================================================================
import ipywidgets as widgets
from ipywidgets import interact, Layout
import pandas as pd
import plotly.graph_objects as go

def run_agri_settlement_calculator(
    cargo_volume_mt=2500,
    price_per_mt=450.0,
    legacy_fx_fee_pct=2.5,
    fx_slippage_pct=1.2,
    settlement_days=7,
    annual_capital_cost_pct=12.0,
    stablecoin_fee_pct=0.2,
    gas_flat_fee=15.0
):
    # Core Financial Calculations
    cargo_value = cargo_volume_mt * price_per_mt

    # Legacy Banking Rail Costs
    legacy_fx_cost = cargo_value * (legacy_fx_fee_pct / 100.0)
    legacy_slippage_cost = cargo_value * (fx_slippage_pct / 100.0)
    legacy_capital_cost = cargo_value * (annual_capital_cost_pct / 100.0) * (settlement_days / 365.0)
    total_legacy_cost = legacy_fx_cost + legacy_slippage_cost + legacy_capital_cost

    # Enterprise Stablecoin Rail Costs
    stablecoin_variable_fee = cargo_value * (stablecoin_fee_pct / 100.0)
    total_stablecoin_cost = stablecoin_variable_fee + gas_flat_fee

    # Net Savings Calculations
    net_savings = total_legacy_cost - total_stablecoin_cost
    savings_margin_pct = (net_savings / cargo_value * 100) if cargo_value > 0 else 0
    cost_reduction_pct = (net_savings / total_legacy_cost * 100) if total_legacy_cost > 0 else 0

    # Print Executive Summary Table
    print("=" * 70)
    print(f" EXPORTER FINANCIAL SUMMARY | Cargo Value: ${cargo_value:,.2f} USD")
    print("=" * 70)
    summary_df = pd.DataFrame({
        'Cost Component': [
            'FX Intermediary Markup',
            'Transit FX Slippage (Volatility Risk)',
            'Working Capital Lockup Cost',
            'Network / Transaction Gas Fees',
            'TOTAL SETTLEMENT COST'
        ],
        'Legacy SWIFT / Wire ($)': [
            f"${legacy_fx_cost:,.2f}",
            f"${legacy_slippage_cost:,.2f}",
            f"${legacy_capital_cost:,.2f}",
            "$0.00",
            f"${total_legacy_cost:,.2f}"
        ],
        'Stablecoin Rail ($)': [
            f"${stablecoin_variable_fee:,.2f}",
            "$0.00 (Instant Settle)",
            "$0.00 (<1 hr Settle)",
            f"${gas_flat_fee:,.2f}",
            f"${total_stablecoin_cost:,.2f}"
        ]
    })
    print(summary_df.to_string(index=False))
    print("-" * 70)
    print(f" NET EXPORTER SAVINGS:       ${net_savings:,.2f} USD")
    print(f" COST REDUCTION RATIO:       {cost_reduction_pct:.1f}% Savings vs Legacy Wire")
    print(f" EFFECTIVE MARGIN EXPANSION: +{savings_margin_pct:.2f}% of Total Gross Cargo Value")
    print("=" * 70)

    # Plot Visual Comparison Breakdown
    fig = go.Figure()

    # Legacy Costs Breakdown Bar
    fig.add_trace(go.Bar(
        name='Legacy SWIFT Rail',
        x=['FX Markup', 'FX Slippage Risk', 'Capital Lockup', 'Total Cost'],
        y=[legacy_fx_cost, legacy_slippage_cost, legacy_capital_cost, total_legacy_cost],
        marker_color='#FF6B6B',
        text=[f"${v:,.0f}" for v in [legacy_fx_cost, legacy_slippage_cost, legacy_capital_cost, total_legacy_cost]],
        textposition='auto'
    ))

    # Stablecoin Costs Breakdown Bar
    fig.add_trace(go.Bar(
        name='Stablecoin Rail',
        x=['FX Markup', 'FX Slippage Risk', 'Capital Lockup', 'Total Cost'],
        y=[stablecoin_variable_fee, 0, 0, total_stablecoin_cost],
        marker_color='#008080',
        text=[f"${v:,.0f}" for v in [stablecoin_variable_fee, 0, 0, total_stablecoin_cost]],
        textposition='auto'
    ))

    fig.update_layout(
        title=f"<b>Settlement Cost Comparison breakdown (${cargo_value:,.0f} Cargo Value)</b>",
        yaxis_title="Total Cost (USD)",
        barmode='group',
        template='plotly_white',
        height=450
    )
    fig.show()

# Interactive Controls Setup
interact(
    run_agri_settlement_calculator,
    cargo_volume_mt=widgets.IntSlider(min=100, max=20000, step=100, value=2500, description='Cargo Vol (MT):', layout=Layout(width='600px')),
    price_per_mt=widgets.FloatSlider(min=50.0, max=3000.0, step=25.0, value=450.0, description='Price/MT ($):', layout=Layout(width='600px')),
    legacy_fx_fee_pct=widgets.FloatSlider(min=0.5, max=5.0, step=0.1, value=2.5, description='Legacy FX %:', layout=Layout(width='600px')),
    fx_slippage_pct=widgets.FloatSlider(min=0.0, max=5.0, step=0.1, value=1.2, description='FX Slippage %:', layout=Layout(width='600px')),
    settlement_days=widgets.IntSlider(min=1, max=14, step=1, value=7, description='Transit Days:', layout=Layout(width='600px')),
    annual_capital_cost_pct=widgets.FloatSlider(min=4.0, max=24.0, step=0.5, value=12.0, description='Capital APR %:', layout=Layout(width='600px')),
    stablecoin_fee_pct=widgets.FloatSlider(min=0.05, max=1.0, step=0.05, value=0.2, description='Stablecoin %:', layout=Layout(width='600px')),
    gas_flat_fee=widgets.FloatSlider(min=1.0, max=100.0, step=1.0, value=15.0, description='Gas Fee ($):', layout=Layout(width='600px'))
);

interactive(children=(IntSlider(value=2500, description='Cargo Vol (MT):', layout=Layout(width='600px'), max=2…

In [ ]:
# Re-running the calculator with default parameters to display the summary table
run_agri_settlement_calculator()

In [7]:
# ==============================================================================
# 1. SETUP & DEPENDENCIES
# ==============================================================================
!pip install -q plotly pandas numpy

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set clean template for all charts
PLOT_TEMPLATE = "plotly_white"
PRIMARY_COLOR = "#008080"  # Teal (Utility / Web3 Ag)
ACCENT_COLOR = "#FF6B6B"   # Coral (Legacy / Speculation)

In [6]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set clean template for all charts
PLOT_TEMPLATE = "plotly_white"
PRIMARY_COLOR = "#008080"  # Teal (Utility / Web3 Ag)
ACCENT_COLOR = "#FF6B6B"   # Coral (Legacy / Speculation)

# ==============================================================================
# 2. MODULE 1: SETTLEMENT FRICTION & WORKING CAPITAL LOCKUP
# ==============================================================================
df_settlement = pd.DataFrame({
    'Settlement Rail': ['Comparison', 'Comparison', 'Comparison'],
    'Metric': ['Avg. Clearance Time (Hours)', 'FX & Intermediary Fee (%)', 'Working Capital Lockup (Days)'],
    'Value': [168, 4.5, 7.0],       # Legacy: 7 days clearance, 4.5% fees, 7 days lockup
    'Value_Alt': [0.08, 0.2, 0.005]  # Stablecoin: 5 mins (0.08h), 0.2% fee, ~0 days lockup
})

# Reshape for side-by-side grouped horizontal comparison
fig1 = go.Figure()

# Use df_settlement directly for plotting
categories_plot = df_settlement['Metric']
legacy_vals_plot = df_settlement['Value']
stablecoin_vals_plot = df_settlement['Value_Alt']

fig1.add_trace(go.Bar(
    y=categories_plot,
    x=legacy_vals_plot,
    name='Legacy Wire / Letter of Credit',
    orientation='h',
    marker_color=ACCENT_COLOR,
    text=[f"{v} hrs" if i==0 else f"{v}%" if i==1 else f"{v} days" for i, v in enumerate(legacy_vals_plot)],
    textposition='auto'
))

fig1.add_trace(go.Bar(
    y=categories_plot,
    x=stablecoin_vals_plot,
    name='Stablecoin Rail',
    orientation='h',
    marker_color=PRIMARY_COLOR,
    text=["5 mins (<0.1h)", "0.2%", "< 1 hour"], # Keep descriptive text as Value_Alt is numeric
    textposition='auto'
))

fig1.update_layout(
    title='<b>Cross-Border B2B Agricultural Settlement: Legacy vs. Stablecoin Rails</b>',
    barmode='group',
    template=PLOT_TEMPLATE,
    height=400,
    xaxis_title="Magnitude (Log Scale for Visual Clarity)",
    xaxis_type="log",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig1.show()

In [8]:
# ==============================================================================
# 3. MODULE 2: RWA TOKENIZED INVOICE & TRADE CREDIT LIQUIDITY
# ==============================================================================
days = np.arange(0, 91, 1)

# Traditional 90-day invoice payment trajectory
traditional_liquidity = np.where(days < 90, 0, 100)

# Tokenized RWA liquidity (Immediate 85% advance on-chain, 15% settlement on completion)
rwa_liquidity = np.where(days == 0, 85, 85)
rwa_liquidity = np.where(days == 90, 100, rwa_liquidity)

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=days, y=traditional_liquidity,
    mode='lines',
    name='Traditional Corporate Invoice (Net-90)',
    line=dict(color=ACCENT_COLOR, width=3, dash='dash')
))

fig2.add_trace(go.Scatter(
    x=days, y=rwa_liquidity,
    mode='lines+markers',
    name='Tokenized RWA Invoice Pool',
    line=dict(color=PRIMARY_COLOR, width=3),
    marker=dict(size=6)
))

fig2.add_annotation(
    x=0, y=85, text="Instant 85% Liquidity via Global Debt Pools",
    showarrow=True, arrowhead=2, ax=140, ay=-30
)

fig2.add_annotation(
    x=90, y=0, text="90-Day Unpaid Waiting Period",
    showarrow=True, arrowhead=2, ax=-100, ay=40
)

fig2.update_layout(
    title='<b>Farmer Liquidity Trajectory: Net-90 Invoicing vs. On-Chain RWA Advance</b>',
    xaxis_title='Days From Harvest / Shipping',
    yaxis_title='Capital Available to Exporter (%)',
    template=PLOT_TEMPLATE,
    height=450,
    yaxis=dict(range=[-5, 110]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig2.show()

In [9]:
# ==============================================================================
# 4. MODULE 3: PARAMETRIC CROP INSURANCE AUTOMATION
# ==============================================================================
np.random.seed(42)
days_season = np.arange(1, 61)

# Rainfall simulation in mm (Drought threshold set at < 5mm cumulative over 10 days)
daily_rainfall = np.random.exponential(scale=3.5, size=60)
daily_rainfall[20:35] = 0.2  # Simulated 15-day severe drought period

cumulative_rainfall_10d = pd.Series(daily_rainfall).rolling(window=10, min_periods=1).sum().values
drought_threshold = 12.0  # Threshold in mm over 10 days

# Smart contract payout state (0 = No payout, 100 = Instant payout executed)
payout_status = np.where(cumulative_rainfall_10d < drought_threshold, 100, 0)

fig3 = make_subplots(specs=[[{"secondary_y": True}]])

# Rainfall Bar Chart
fig3.add_trace(
    go.Bar(x=days_season, y=daily_rainfall, name="Daily Rainfall (IoT Sensor Data)", marker_color="#4A90E2", opacity=0.6),
    secondary_y=False
)

# 10-Day Rolling Cumulative
fig3.add_trace(
    go.Scatter(x=days_season, y=cumulative_rainfall_10d, name="10-Day Rolling Rainfall", line=dict(color="#2C3E50", width=2)),
    secondary_y=False
)

# Threshold Line
fig3.add_trace(
    go.Scatter(x=[1, 60], y=[drought_threshold, drought_threshold], name="Drought Breach Threshold", line=dict(color=ACCENT_COLOR, width=2, dash='dash')),
    secondary_y=False
)

# Smart Contract Instant Payout Execution
fig3.add_trace(
    go.Scatter(x=days_season, y=payout_status, name="Smart Contract Payout ($)", line=dict(color=PRIMARY_COLOR, width=3)),
    secondary_y=True
)

fig3.update_layout(
    title='<b>Parametric Crop Insurance: IoT Weather Triggers vs. Smart Contract Liquidity Payout</b>',
    xaxis_title='Growing Season Days',
    template=PLOT_TEMPLATE,
    height=480,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig3.update_yaxes(title_text="Rainfall (mm)", secondary_y=False)
fig3.update_yaxes(title_text="Insurance Payout Executed (%)", secondary_y=True, range=[-10, 120])

fig3.show()

In [10]:
# ==============================================================================
# 5. MODULE 4: PARADIGM SHIFT COMPARISON MATRIX
# ==============================================================================
paradigm_data = {
    'Feature': ['Underlying Asset', 'Settlement Speed', 'Primary Value Driver', 'Target Participants'],
    'Speculative Crypto': ['Unbacked tokens, volatile code', 'High-friction on/off ramps', 'Market hype & speculation', 'Retail speculators'],
    'Agri Web3 Infrastructure': ['Physical food cargo, land, invoices', 'Instant B2B stablecoin rails', 'Real-world yield & liquidity', 'Enterprise exporters & institutions']
}

df_paradigm = pd.DataFrame(paradigm_data)

fig4 = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Dimension</b>', '<b>Speculative Crypto</b>', '<b>Agricultural Web3 Infrastructure</b>'],
        fill_color=PRIMARY_COLOR,
        align='left',
        font=dict(color='white', size=13)
    ),
    cells=dict(
        values=[df_paradigm['Feature'], df_paradigm['Speculative Crypto'], df_paradigm['Agri Web3 Infrastructure']],
        fill_color=[['#F9F9F9', '#FFFFFF']*2],
        align='left',
        font=dict(size=12),
        height=35
    )
)])

fig4.update_layout(
    title='<b>Paradigm Shift: Speculation vs. Real-World Utility in Agriculture</b>',
    height=300,
    margin=dict(l=10, r=10, t=50, b=10)
)

fig4.show()